In [2]:
%pip install -U langchain langchain-groq langgraph langchain-community ddgs pandas

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 26.2.1 -> 26.3
[notice] To update, run: python.exe -m pip install --upgrade pip


In [3]:
import os
from getpass import getpass

os.environ.pop("GROQ_API_KEY", None)

new_key = getpass("Enter your new Groq API key: ")

os.environ["GROQ_API_KEY"] = new_key.strip()

print("API key loaded:", bool(os.environ["GROQ_API_KEY"]))

API key loaded: True


In [4]:
from groq import Groq

client = Groq(
    api_key=os.environ["GROQ_API_KEY"]
)

response = client.chat.completions.create(
    model="openai/gpt-oss-20b",
    messages=[
        {"role": "user", "content": "Say hello"}
    ]
)

print(response.choices[0].message.content)

Hello! 😄


In [5]:
from langchain_groq import ChatGroq

llm = ChatGroq(
    model="openai/gpt-oss-20b",
    temperature=0,
    api_key=os.environ["GROQ_API_KEY"]
)

print(llm.invoke("Hello").content)

Hello! How can I help you today?


In [6]:
import sqlite3

DB_NAME = "lab_booking.db"

def get_connection():
    conn = sqlite3.connect(DB_NAME)
    conn.execute("PRAGMA foreign_keys = ON")
    return conn

conn = get_connection()
cursor = conn.cursor()

cursor.execute("""
CREATE TABLE IF NOT EXISTS tests (
    test_id INTEGER PRIMARY KEY,
    test_name TEXT UNIQUE NOT NULL,
    category TEXT NOT NULL,
    price REAL NOT NULL
)
""")

cursor.execute("""
CREATE TABLE IF NOT EXISTS slots (
    slot_id INTEGER PRIMARY KEY,
    test_id INTEGER NOT NULL,
    slot_time TEXT NOT NULL,
    is_booked INTEGER DEFAULT 0,
    FOREIGN KEY(test_id) REFERENCES tests(test_id)
)
""")

cursor.execute("""
CREATE TABLE IF NOT EXISTS bookings (
    booking_id INTEGER PRIMARY KEY AUTOINCREMENT,
    user_id TEXT NOT NULL,
    test_id INTEGER NOT NULL,
    slot_id INTEGER NOT NULL,
    status TEXT DEFAULT 'confirmed',
    FOREIGN KEY(test_id) REFERENCES tests(test_id),
    FOREIGN KEY(slot_id) REFERENCES slots(slot_id),
    UNIQUE(user_id, test_id, slot_id)
)
""")

conn.commit()
conn.close()

print("Database and tables created successfully!")

Database and tables created successfully!


In [7]:
tests_data = [
    ("CBC", "Blood", 350),
    ("Blood Sugar", "Blood", 200),
    ("Lipid Profile", "Blood", 800),
    ("Liver Function Test", "Blood", 900),
    ("Thyroid Test", "Hormone", 600),
    ("Vitamin D", "Vitamin", 1200),
    ("Vitamin B12", "Vitamin", 700),
    ("HbA1c", "Blood", 500)
]

conn = get_connection()
cursor = conn.cursor()

for test in tests_data:
    cursor.execute("""
        INSERT OR IGNORE INTO tests
        (test_name, category, price)
        VALUES (?, ?, ?)
    """, test)

cursor.execute("SELECT test_id FROM tests ORDER BY test_id")
test_ids = [row[0] for row in cursor.fetchall()]

for test_id in test_ids:
    for i in range(3):
        cursor.execute("""
            INSERT OR IGNORE INTO slots
            (test_id, slot_time, is_booked)
            VALUES (?, ?, 0)
        """, (
            test_id,
            f"2026-11-{10+i:02d} 09:00"
        ))

# Make one test have no free slots
cursor.execute("""
    UPDATE slots SET is_booked = 1
    WHERE test_id = ?
""", (test_ids[-1],))

conn.commit()
conn.close()

print("Sample tests and appointment slots inserted!")

Sample tests and appointment slots inserted!


In [8]:
conn = get_connection()

print("TESTS")
display(__import__("pandas").read_sql_query(
    "SELECT * FROM tests", conn
))

print("SLOTS")
display(__import__("pandas").read_sql_query(
    "SELECT * FROM slots", conn
))

conn.close()

TESTS


,test_id,test_name,category,price
0,1,CBC,Blood,350.0
1,2,Blood Sugar,Blood,200.0
2,3,Lipid Profile,Blood,800.0
3,4,Liver Function Test,Blood,900.0
4,5,Thyroid Test,Hormone,600.0
5,6,Vitamin D,Vitamin,1200.0
6,7,Vitamin B12,Vitamin,700.0
7,8,HbA1c,Blood,500.0


SLOTS


,slot_id,test_id,slot_time,is_booked
0,1,1,2026-11-10 09:00,0
1,2,1,2026-11-11 09:00,0
2,3,1,2026-11-12 09:00,0
3,4,2,2026-11-10 09:00,0
4,5,2,2026-11-11 09:00,0
...,...,...,...,...
139,140,7,2026-11-11 09:00,0
140,141,7,2026-11-12 09:00,0
141,142,8,2026-11-10 09:00,1
142,143,8,2026-11-11 09:00,1


In [9]:
conn = get_connection()

print("TESTS")
display(__import__("pandas").read_sql_query(
    "SELECT * FROM tests", conn
))

print("SLOTS")
display(__import__("pandas").read_sql_query(
    "SELECT * FROM slots", conn
))

conn.close()


TESTS


,test_id,test_name,category,price
0,1,CBC,Blood,350.0
1,2,Blood Sugar,Blood,200.0
2,3,Lipid Profile,Blood,800.0
3,4,Liver Function Test,Blood,900.0
4,5,Thyroid Test,Hormone,600.0
5,6,Vitamin D,Vitamin,1200.0
6,7,Vitamin B12,Vitamin,700.0
7,8,HbA1c,Blood,500.0


SLOTS


,slot_id,test_id,slot_time,is_booked
0,1,1,2026-11-10 09:00,0
1,2,1,2026-11-11 09:00,0
2,3,1,2026-11-12 09:00,0
3,4,2,2026-11-10 09:00,0
4,5,2,2026-11-11 09:00,0
...,...,...,...,...
139,140,7,2026-11-11 09:00,0
140,141,7,2026-11-12 09:00,0
141,142,8,2026-11-10 09:00,1
142,143,8,2026-11-11 09:00,1


In [10]:
from langchain.tools import tool

@tool
def list_tests():
    """List all available diagnostic tests with category and price."""

    conn = get_connection()

    rows = conn.execute("""
        SELECT test_id, test_name, category, price
        FROM tests
        ORDER BY test_id
    """).fetchall()

    conn.close()

    return str(rows)


@tool
def check_slots(test_id: int):
    """Check available appointment slots for a test ID."""

    conn = get_connection()

    rows = conn.execute("""
        SELECT slot_id, slot_time
        FROM slots
        WHERE test_id = ? AND is_booked = 0
        ORDER BY slot_time
    """, (test_id,)).fetchall()

    conn.close()

    if not rows:
        return "No available slots for this test."

    return str(rows)

In [11]:
from dataclasses import dataclass
from langchain.tools import ToolRuntime

@dataclass
class UserContext:
    user_id: str

In [12]:
user1 = UserContext(user_id="patient_101")
user2 = UserContext(user_id="patient_202")

print(user1)
print(user2)

UserContext(user_id='patient_101')
UserContext(user_id='patient_202')


In [13]:
@tool
def my_bookings(runtime: ToolRuntime[UserContext]):
    """Show bookings belonging to the current patient."""

    user_id = runtime.context.user_id

    conn = get_connection()

    rows = conn.execute("""
        SELECT b.booking_id, t.test_name,
               s.slot_time, b.status
        FROM bookings b
        JOIN tests t ON b.test_id = t.test_id
        JOIN slots s ON b.slot_id = s.slot_id
        WHERE b.user_id = ?
        ORDER BY b.booking_id
    """, (user_id,)).fetchall()

    conn.close()

    return str(rows) if rows else "No bookings found."

In [14]:
@tool
def book_test(test_id: int, slot_id: int,
              runtime: ToolRuntime[UserContext]):
    """Book an available test appointment for the current patient."""

    user_id = runtime.context.user_id

    conn = get_connection()

    try:
        conn.execute("BEGIN IMMEDIATE")

        test = conn.execute(
            "SELECT test_name FROM tests WHERE test_id = ?",
            (test_id,)
        ).fetchone()

        if not test:
            conn.rollback()
            return "Invalid test ID."

        slot = conn.execute("""
            SELECT slot_time FROM slots
            WHERE slot_id = ? AND test_id = ? AND is_booked = 0
        """, (slot_id, test_id)).fetchone()

        if not slot:
            conn.rollback()
            return "This slot is unavailable."

        duplicate = conn.execute("""
            SELECT booking_id FROM bookings
            WHERE user_id = ? AND test_id = ?
            AND status = 'confirmed'
        """, (user_id, test_id)).fetchone()

        if duplicate:
            conn.rollback()
            return "You already have a confirmed booking for this test."

        cursor = conn.execute("""
            INSERT INTO bookings(user_id, test_id, slot_id)
            VALUES (?, ?, ?)
        """, (user_id, test_id, slot_id))

        conn.execute("""
            UPDATE slots SET is_booked = 1
            WHERE slot_id = ?
        """, (slot_id,))

        booking_id = cursor.lastrowid

        # Read receipt directly from database
        receipt = conn.execute("""
            SELECT b.booking_id, b.user_id, t.test_name,
                   t.price, s.slot_time, b.status
            FROM bookings b
            JOIN tests t ON b.test_id = t.test_id
            JOIN slots s ON b.slot_id = s.slot_id
            WHERE b.booking_id = ?
        """, (booking_id,)).fetchone()

        conn.commit()

        return f"Booking successful. Database receipt: {receipt}"

    except Exception as e:
        conn.rollback()
        return f"Booking failed: {str(e)}"

    finally:
        conn.close()

In [15]:
@tool
def cancel_booking(booking_id: int,
                   runtime: ToolRuntime[UserContext]):
    """Cancel a booking belonging to the current patient."""

    user_id = runtime.context.user_id

    conn = get_connection()

    try:
        conn.execute("BEGIN IMMEDIATE")

        booking = conn.execute("""
            SELECT slot_id FROM bookings
            WHERE booking_id = ?
            AND user_id = ?
            AND status = 'confirmed'
        """, (booking_id, user_id)).fetchone()

        if not booking:
            conn.rollback()
            return "Booking not found or does not belong to you."

        slot_id = booking[0]

        conn.execute("""
            UPDATE bookings SET status = 'cancelled'
            WHERE booking_id = ? AND user_id = ?
        """, (booking_id, user_id))

        conn.execute("""
            UPDATE slots SET is_booked = 0
            WHERE slot_id = ?
        """, (slot_id,))

        conn.commit()

        return f"Booking {booking_id} cancelled successfully."

    except Exception as e:
        conn.rollback()
        return f"Cancellation failed: {str(e)}"

    finally:
        conn.close()

In [16]:
for tool_item in [
    list_tests,
    check_slots,
    my_bookings,
    book_test,
    cancel_booking
]:
    print("\nTool:", tool_item.name)

    # Show only arguments exposed to the AI model
    print("Schema:", tool_item.tool_call_schema.model_json_schema())


Tool: list_tests
Schema: {'description': 'List all available diagnostic tests with category and price.', 'properties': {}, 'title': 'list_tests', 'type': 'object'}

Tool: check_slots
Schema: {'description': 'Check available appointment slots for a test ID.', 'properties': {'test_id': {'title': 'Test Id', 'type': 'integer'}}, 'required': ['test_id'], 'title': 'check_slots', 'type': 'object'}

Tool: my_bookings
Schema: {'description': 'Show bookings belonging to the current patient.', 'properties': {}, 'title': 'my_bookings', 'type': 'object'}

Tool: book_test
Schema: {'description': 'Book an available test appointment for the current patient.', 'properties': {'test_id': {'title': 'Test Id', 'type': 'integer'}, 'slot_id': {'title': 'Slot Id', 'type': 'integer'}}, 'required': ['test_id', 'slot_id'], 'title': 'book_test', 'type': 'object'}

Tool: cancel_booking
Schema: {'description': 'Cancel a booking belonging to the current patient.', 'properties': {'booking_id': {'title': 'Booking Id'

In [17]:
from langchain_community.tools import DuckDuckGoSearchRun

search = DuckDuckGoSearchRun()

@tool
def general_health_search(query: str):
    """Search general information about laboratory tests and preparation.
    Never use this tool for prices, availability or bookings."""

    return search.run(query)

C:\Users\prach\AppData\Local\Temp\ipykernel_27436\2027050327.py:1: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.tools import DuckDuckGoSearchRun


In [18]:
print(general_health_search.invoke({
    "query": "What does a CBC blood test measure?"
}))

Mar 13, 2026 · A CBC blood test measures your red cells, white cells, and platelets to give doctors a broad picture of your overall health in one simple draw. In a CBC for red blood cells, your providers look at: 1. Red blood count (RBC).This is the number of red blood cells in a certain amount of blood, usually a microliter (mcL). For reference, one drop of blood is about 40 mcL to 50 mcL. 2. Hemoglobin (Hb).This is the amount of hemoglobin in your blood. 3. Hematocrit (Hct).This is the percentage of yo... See full list on my.clevelandclinic.org The results of your platelet count on a CBC include: 1. Platelet count.The total number of platelets in your blood sample. 2. Mean platelet volume (MPV).MPV is your average platelet size. See full list on my.clevelandclinic.org Your provider can order a CBC with just total white blood cell counts or with a differential. A differential counts each different type of white blood cell and immature red blood cells. These might be shown as an absolu

In [19]:
SYSTEM_PROMPT = """
You are a diagnostic laboratory booking assistant.

RULES:

1. Use the database tools for all test prices.
2. Use the database tools for all appointment availability.
3. Never guess prices or available slots.
4. Always confirm the test and slot with the patient before booking.
5. Never book without human approval.
6. Never cancel without human approval.
7. Only show bookings belonging to the current user.
8. Use web search only for general health information.
9. Never diagnose diseases.
10. Never recommend or select a test based on symptoms.
11. For symptom-related questions, explain that you cannot diagnose
    and advise the patient to consult a qualified healthcare professional.
12. Provide booking receipts using database results.
"""

In [20]:
from langchain.agents import create_agent
from langgraph.checkpoint.memory import InMemorySaver

tools = [
    list_tests,
    check_slots,
    my_bookings,
    book_test,
    cancel_booking,
    general_health_search
]

memory = InMemorySaver()

agent = create_agent(
    model=llm,
    tools=tools,
    system_prompt=SYSTEM_PROMPT,
    context_schema=UserContext,
    checkpointer=memory
)

print("Lab Booking Agent created successfully!")

Lab Booking Agent created successfully!


In [21]:
config = {
    "configurable": {
        "thread_id": "patient101_chat"
    }
}

print("Conversation configuration created!")

Conversation configuration created!


In [22]:
user1 = UserContext(user_id="patient_101")

print(user1)

UserContext(user_id='patient_101')


In [23]:
response = agent.invoke(
    {
        "messages": [
            {
                "role": "user",
                "content": "Which of those tests is cheapest?"
            }
        ]
    },
    config=config,
    context=user1
)

print(response["messages"][-1].content)


The cheapest test on our list is the **Blood Sugar** test, priced at **₹200.00**.


In [24]:
%pip install -U "langchain>=1.0" "langgraph>=1.0"

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 26.2.1 -> 26.3
[notice] To update, run: python.exe -m pip install --upgrade pip


In [25]:
from langchain.agents.middleware import HumanInTheLoopMiddleware

approval_middleware = HumanInTheLoopMiddleware(
    interrupt_on={
        "book_test": {
            "allowed_decisions": ["approve", "reject"]
        },
        "cancel_booking": {
            "allowed_decisions": ["approve", "reject"]
        }
    }
)

agent = create_agent(
    model=llm,
    tools=tools,
    system_prompt=SYSTEM_PROMPT,
    context_schema=UserContext,
    middleware=[approval_middleware],
    checkpointer=memory
)

print("Approval-enabled agent created!")

Approval-enabled agent created!


In [26]:
from langgraph.types import Command

approval_config = {
    "configurable": {
        "thread_id": "approval_test_101"
    }
}

result = agent.invoke(
    {
        "messages": [
            {
                "role": "user",
                "content": "Book test ID 1 in slot ID 1."
            }
        ]
    },
    config=approval_config,
    context=user1,
    version="v2"
)

if result.interrupts:
    for item in result.interrupts:
        print("Approval request:", item.value)

In [27]:
result = agent.invoke(
    Command(
        resume={
            "decisions": [
                {"type": "approve"}
            ]
        }
    ),
    config=approval_config,
    context=user1,
    version="v2"
)

print(result)

GraphOutput(value={'messages': [HumanMessage(content='Book test ID 1 in slot ID 1.', additional_kwargs={}, response_metadata={}, id='878a87f5-8339-40e6-8698-d429de38152f'), AIMessage(content='', additional_kwargs={'reasoning_content': "We need to follow rules. The user wants to book test ID 1 in slot ID 1. We must check availability first. Use check_slots for test_id 1. Then confirm with user. Also we must not book without human approval. So we need to confirm. Also we must not guess. So we need to check slots. Let's call check_slots.", 'tool_calls': [{'id': 'fc_7f2c7183-3756-4ac3-a0f3-e156adeef4f5', 'function': {'arguments': '{"test_id":1}', 'name': 'check_slots'}, 'type': 'function'}]}, response_metadata={'token_usage': {'completion_tokens': 99, 'prompt_tokens': 416, 'total_tokens': 515, 'completion_time': 0.104218857, 'completion_tokens_details': {'reasoning_tokens': 75}, 'prompt_time': 0.028571561, 'prompt_tokens_details': None, 'queue_time': 0.289928734, 'total_time': 0.132790418}

In [28]:
conn = get_connection()

before = conn.execute(
    "SELECT COUNT(*) FROM bookings"
).fetchone()[0]

conn.close()

print("Bookings before:", before)

Bookings before: 1


In [29]:
conn = get_connection()

after = conn.execute(
    "SELECT COUNT(*) FROM bookings"
).fetchone()[0]

conn.close()

print("Bookings after:", after)
print("Rejected request changed database:", before != after)

Bookings after: 1
Rejected request changed database: False


In [30]:
failure_count = {"value": 0}

def reliable_check_slots(test_id):

    for attempt in range(1, 3):
        try:
            print("Attempt:", attempt)

            if failure_count["value"] == 0:
                failure_count["value"] += 1
                raise RuntimeError("Simulated temporary database failure")

            conn = get_connection()

            rows = conn.execute("""
                SELECT slot_id, slot_time
                FROM slots
                WHERE test_id = ? AND is_booked = 0
            """, (test_id,)).fetchall()

            conn.close()

            return rows

        except Exception as error:
            print("Error:", error)

            if attempt == 2:
                raise

    return []

In [31]:
print(reliable_check_slots(1))

Attempt: 1
Error: Simulated temporary database failure
Attempt: 2
[(1, '2026-11-10 09:00'), (2, '2026-11-11 09:00'), (3, '2026-11-12 09:00'), (25, '2026-11-10 09:00'), (26, '2026-11-11 09:00'), (27, '2026-11-12 09:00'), (49, '2026-11-10 09:00'), (50, '2026-11-11 09:00'), (51, '2026-11-12 09:00'), (73, '2026-11-10 09:00'), (74, '2026-11-11 09:00'), (75, '2026-11-12 09:00'), (97, '2026-11-10 09:00'), (98, '2026-11-11 09:00'), (99, '2026-11-12 09:00'), (121, '2026-11-10 09:00'), (122, '2026-11-11 09:00'), (123, '2026-11-12 09:00')]


In [32]:
import re

def redact_email(message):

    pattern = r'[\w.+-]+@[\w.-]+\.[a-zA-Z]{2,}'

    return re.sub(
        pattern,
        "[REDACTED EMAIL]",
        message
    )

In [33]:
original_message = (
    "Please book my CBC test. My email is "
    "prachi@example.com"
)

safe_message = redact_email(original_message)

print("Original:", original_message)
print("Sent to model:", safe_message)

Original: Please book my CBC test. My email is prachi@example.com
Sent to model: Please book my CBC test. My email is [REDACTED EMAIL]


In [34]:
def show_agent_response(result):

    for message in result["messages"]:
        if message.type == "ai":
            for call in message.tool_calls:
                print("TOOL CALL:", call["name"])
                print("ARGUMENTS:", call["args"])

    print("\nASSISTANT:")
    print(result["messages"][-1].content)

In [35]:
from langgraph.types import Command

def run_chatbot(user_id):

    context = UserContext(user_id=user_id)
    thread_id = f"chat_{user_id}"

    config = {
        "configurable": {
            "thread_id": thread_id
        }
    }

    print("\nWelcome to Lab Booking Assistant!")
    print("User:", user_id)
    print("Type exit to stop.")

    while True:

        message = input("\nYou: ")

        if message.lower() == "exit":
            print("Chat ended.")
            break

        safe_message = redact_email(message)

        result = agent.invoke(
            {
                "messages": [
                    {"role": "user", "content": safe_message}
                ]
            },
            config=config,
            context=context,
            version="v2"
        )

        while result.interrupts:

            for item in result.interrupts:
                print("\nAPPROVAL REQUIRED:")
                print(item.value)

            decision = input(
                "Enter approve or reject: "
            ).strip().lower()

            if decision not in ["approve", "reject"]:
                print("Invalid choice. Enter approve or reject.")
                continue

            result = agent.invoke(
                Command(
                    resume={
                        "decisions": [
                            {"type": decision}
                        ]
                    }
                ),
                config=config,
                context=context,
                version="v2"
            )

        show_agent_response(result)

In [36]:
run_chatbot("patient_101")


Welcome to Lab Booking Assistant!
User: patient_101
Type exit to stop.
TOOL CALL: list_tests
ARGUMENTS: {}

ASSISTANT:
Here are all the diagnostic tests we currently offer:

| Test ID | Test Name          | Category | Price (₹) |
|---------|--------------------|----------|-----------|
| 1       | CBC                | Blood    | 350.0     |
| 2       | Blood Sugar        | Blood    | 200.0     |
| 3       | Lipid Profile      | Blood    | 800.0     |
| 4       | Liver Function Test | Blood  | 900.0     |
| 5       | Thyroid Test       | Hormone  | 600.0     |
| 6       | Vitamin D          | Vitamin  | 1200.0    |
| 7       | Vitamin B12        | Vitamin  | 700.0     |
| 8       | HbA1c              | Blood    | 500.0     |

Let me know which test(s) you’d like to book, and I’ll check the available appointment slots for you.


C:\Users\prach\AppData\Local\Temp\ipykernel_27436\2080025358.py:3: LangGraphDeprecatedSinceV11: Accessing GraphOutput via `result[key]` is deprecated. Use `result.value` to access the output value directly, or `result.interrupts` for interrupts. Deprecated in LangGraph V1.1 to be removed in V3.0.
  for message in result["messages"]:
C:\Users\prach\AppData\Local\Temp\ipykernel_27436\2080025358.py:10: LangGraphDeprecatedSinceV11: Accessing GraphOutput via `result[key]` is deprecated. Use `result.value` to access the output value directly, or `result.interrupts` for interrupts. Deprecated in LangGraph V1.1 to be removed in V3.0.
  print(result["messages"][-1].content)


TOOL CALL: list_tests
ARGUMENTS: {}
TOOL CALL: check_slots
ARGUMENTS: {'test_id': 5}

ASSISTANT:
Great! The **Thyroid Test** (Test ID 5) is available on the following dates and times:

| Slot ID | Date & Time          |
|---------|----------------------|
| 13      | 2026‑11‑10 09:00 AM |
| 37      | 2026‑11‑10 09:00 AM |
| 61      | 2026‑11‑10 09:00 AM |
| 85      | 2026‑11‑10 09:00 AM |
| 109     | 2026‑11‑10 09:00 AM |
| 133     | 2026‑11‑10 09:00 AM |
| 14      | 2026‑11‑11 09:00 AM |
| 38      | 2026‑11‑11 09:00 AM |
| 62      | 2026‑11‑11 09:00 AM |
| 86      | 2026‑11‑11 09:00 AM |
| 110     | 2026‑11‑11 09:00 AM |
| 134     | 2026‑11‑11 09:00 AM |
| 15      | 2026‑11‑12 09:00 AM |
| 39      | 2026‑11‑12 09:00 AM |
| 63      | 2026‑11‑12 09:00 AM |
| 87      | 2026‑11‑12 09:00 AM |
| 111     | 2026‑11‑12 09:00 AM |
| 135     | 2026‑11‑12 09:00 AM |

Please let me know which **date/time** (and corresponding Slot ID) you’d like to book. Once you confirm, I’ll proceed with the reser

C:\Users\prach\AppData\Local\Temp\ipykernel_27436\2080025358.py:3: LangGraphDeprecatedSinceV11: Accessing GraphOutput via `result[key]` is deprecated. Use `result.value` to access the output value directly, or `result.interrupts` for interrupts. Deprecated in LangGraph V1.1 to be removed in V3.0.
  for message in result["messages"]:
C:\Users\prach\AppData\Local\Temp\ipykernel_27436\2080025358.py:10: LangGraphDeprecatedSinceV11: Accessing GraphOutput via `result[key]` is deprecated. Use `result.value` to access the output value directly, or `result.interrupts` for interrupts. Deprecated in LangGraph V1.1 to be removed in V3.0.
  print(result["messages"][-1].content)


TOOL CALL: list_tests
ARGUMENTS: {}
TOOL CALL: check_slots
ARGUMENTS: {'test_id': 5}
TOOL CALL: check_slots
ARGUMENTS: {'test_id': 6}

ASSISTANT:
Here are the available appointment slots for the **Vitamin D** test (Test ID 6):

| Slot ID | Date & Time          |
|---------|----------------------|
| 16      | 2026‑11‑10 09:00 AM |
| 40      | 2026‑11‑10 09:00 AM |
| 64      | 2026‑11‑10 09:00 AM |
| 88      | 2026‑11‑10 09:00 AM |
| 112     | 2026‑11‑10 09:00 AM |
| 136     | 2026‑11‑10 09:00 AM |
| 17      | 2026‑11‑11 09:00 AM |
| 41      | 2026‑11‑11 09:00 AM |
| 65      | 2026‑11‑11 09:00 AM |
| 89      | 2026‑11‑11 09:00 AM |
| 113     | 2026‑11‑11 09:00 AM |
| 137     | 2026‑11‑11 09:00 AM |
| 18      | 2026‑11‑12 09:00 AM |
| 42      | 2026‑11‑12 09:00 AM |
| 66      | 2026‑11‑12 09:00 AM |
| 90      | 2026‑11‑12 09:00 AM |
| 114     | 2026‑11‑12 09:00 AM |
| 138     | 2026‑11‑12 09:00 AM |

Please let me know which **date/time** (and the corresponding Slot ID) you’d like to book

C:\Users\prach\AppData\Local\Temp\ipykernel_27436\2080025358.py:3: LangGraphDeprecatedSinceV11: Accessing GraphOutput via `result[key]` is deprecated. Use `result.value` to access the output value directly, or `result.interrupts` for interrupts. Deprecated in LangGraph V1.1 to be removed in V3.0.
  for message in result["messages"]:
C:\Users\prach\AppData\Local\Temp\ipykernel_27436\2080025358.py:10: LangGraphDeprecatedSinceV11: Accessing GraphOutput via `result[key]` is deprecated. Use `result.value` to access the output value directly, or `result.interrupts` for interrupts. Deprecated in LangGraph V1.1 to be removed in V3.0.
  print(result["messages"][-1].content)


Chat ended.


In [37]:
import pandas as pd

evaluation_results = [
    {
        "Scenario": "Check availability",
        "Expected Tools": "list_tests, check_slots",
        "Actual Tools": "Pending",
        "Expected DB Change": "No change",
        "Actual DB Change": "Pending",
        "Result": "Pending"
    },
    {
        "Scenario": "Booking approved",
        "Expected Tools": "book_test",
        "Actual Tools": "Pending",
        "Expected DB Change": "New booking",
        "Actual DB Change": "Pending",
        "Result": "Pending"
    },
    {
        "Scenario": "Second booking",
        "Expected Tools": "book_test",
        "Actual Tools": "Pending",
        "Expected DB Change": "New booking",
        "Actual DB Change": "Pending",
        "Result": "Pending"
    },
    {
        "Scenario": "Booking rejected",
        "Expected Tools": "book_test",
        "Actual Tools": "Pending",
        "Expected DB Change": "No change",
        "Actual DB Change": "Pending",
        "Result": "Pending"
    },
    {
        "Scenario": "Slot conflict",
        "Expected Tools": "check_slots, book_test",
        "Actual Tools": "Pending",
        "Expected DB Change": "No new booking",
        "Actual DB Change": "Pending",
        "Result": "Pending"
    },
    {
        "Scenario": "Cancellation",
        "Expected Tools": "cancel_booking",
        "Actual Tools": "Pending",
        "Expected DB Change": "Booking cancelled",
        "Actual DB Change": "Pending",
        "Result": "Pending"
    },
    {
        "Scenario": "General information",
        "Expected Tools": "general_health_search",
        "Actual Tools": "Pending",
        "Expected DB Change": "No change",
        "Actual DB Change": "Pending",
        "Result": "Pending"
    },
    {
        "Scenario": "Symptom question",
        "Expected Tools": "No diagnostic tool",
        "Actual Tools": "Pending",
        "Expected DB Change": "No change",
        "Actual DB Change": "Pending",
        "Result": "Pending"
    }
]

evaluation_df = pd.DataFrame(evaluation_results)

display(evaluation_df)

,Scenario,Expected Tools,Actual Tools,Expected DB Change,Actual DB Change,Result
0,Check availability,"list_tests, check_slots",Pending,No change,Pending,Pending
1,Booking approved,book_test,Pending,New booking,Pending,Pending
2,Second booking,book_test,Pending,New booking,Pending,Pending
3,Booking rejected,book_test,Pending,No change,Pending,Pending
4,Slot conflict,"check_slots, book_test",Pending,No new booking,Pending,Pending
5,Cancellation,cancel_booking,Pending,Booking cancelled,Pending,Pending
6,General information,general_health_search,Pending,No change,Pending,Pending
7,Symptom question,No diagnostic tool,Pending,No change,Pending,Pending


In [38]:
run_chatbot("patient_102")


Welcome to Lab Booking Assistant!
User: patient_102
Type exit to stop.
Chat ended.


In [ ]:
run_chatbot("patient_103")


Welcome to Lab Booking Assistant!
User: patient_103
Type exit to stop.
TOOL CALL: list_tests
ARGUMENTS: {}

ASSISTANT:
Here are all the diagnostic tests we offer:

| Test ID | Test Name          | Category | Price (₹) |
|---------|--------------------|----------|-----------|
| 1       | CBC                | Blood    | 350.0     |
| 2       | Blood Sugar        | Blood    | 200.0     |
| 3       | Lipid Profile      | Blood    | 800.0     |
| 4       | Liver Function Test| Blood    | 900.0     |
| 5       | Thyroid Test       | Hormone  | 600.0     |
| 6       | Vitamin D          | Vitamin  | 1200.0    |
| 7       | Vitamin B12        | Vitamin  | 700.0     |
| 8       | HbA1c              | Blood    | 500.0     |

Let me know if you’d like to check availability for a specific test or book an appointment!


C:\Users\prach\AppData\Local\Temp\ipykernel_27436\2080025358.py:3: LangGraphDeprecatedSinceV11: Accessing GraphOutput via `result[key]` is deprecated. Use `result.value` to access the output value directly, or `result.interrupts` for interrupts. Deprecated in LangGraph V1.1 to be removed in V3.0.
  for message in result["messages"]:
C:\Users\prach\AppData\Local\Temp\ipykernel_27436\2080025358.py:10: LangGraphDeprecatedSinceV11: Accessing GraphOutput via `result[key]` is deprecated. Use `result.value` to access the output value directly, or `result.interrupts` for interrupts. Deprecated in LangGraph V1.1 to be removed in V3.0.
  print(result["messages"][-1].content)


TOOL CALL: list_tests
ARGUMENTS: {}
TOOL CALL: check_slots
ARGUMENTS: {'test_id': 6}

ASSISTANT:
Great! The Vitamin D test (Test ID 6) is available on the following dates and times:

| Slot ID | Date & Time          |
|---------|----------------------|
| 16      | 2026‑11‑10 09:00 AM  |
| 40      | 2026‑11‑10 09:00 AM  |
| 64      | 2026‑11‑10 09:00 AM  |
| 88      | 2026‑11‑10 09:00 AM  |
| 112     | 2026‑11‑10 09:00 AM  |
| 136     | 2026‑11‑10 09:00 AM  |
| 17      | 2026‑11‑11 09:00 AM  |
| 41      | 2026‑11‑11 09:00 AM  |
| 65      | 2026‑11‑11 09:00 AM  |
| 89      | 2026‑11‑11 09:00 AM  |
| 113     | 2026‑11‑11 09:00 AM  |
| 137     | 2026‑11‑11 09:00 AM  |
| 18      | 2026‑11‑12 09:00 AM  |
| 42      | 2026‑11‑12 09:00 AM  |
| 66      | 2026‑11‑12 09:00 AM  |
| 90      | 2026‑11‑12 09:00 AM  |
| 114     | 2026‑11‑12 09:00 AM  |
| 138     | 2026‑11‑12 09:00 AM  |

Please let me know which **Slot ID** (or date/time) you’d like to book, and I’ll confirm the appointment for you.


C:\Users\prach\AppData\Local\Temp\ipykernel_27436\2080025358.py:3: LangGraphDeprecatedSinceV11: Accessing GraphOutput via `result[key]` is deprecated. Use `result.value` to access the output value directly, or `result.interrupts` for interrupts. Deprecated in LangGraph V1.1 to be removed in V3.0.
  for message in result["messages"]:
C:\Users\prach\AppData\Local\Temp\ipykernel_27436\2080025358.py:10: LangGraphDeprecatedSinceV11: Accessing GraphOutput via `result[key]` is deprecated. Use `result.value` to access the output value directly, or `result.interrupts` for interrupts. Deprecated in LangGraph V1.1 to be removed in V3.0.
  print(result["messages"][-1].content)


TOOL CALL: list_tests
ARGUMENTS: {}
TOOL CALL: check_slots
ARGUMENTS: {'test_id': 6}

ASSISTANT:
You’d like to book the Vitamin D test on **2026‑11‑11 at 09:00 AM** (Slot ID 17).  
Is that the appointment you’d like to confirm?


C:\Users\prach\AppData\Local\Temp\ipykernel_27436\2080025358.py:3: LangGraphDeprecatedSinceV11: Accessing GraphOutput via `result[key]` is deprecated. Use `result.value` to access the output value directly, or `result.interrupts` for interrupts. Deprecated in LangGraph V1.1 to be removed in V3.0.
  for message in result["messages"]:
C:\Users\prach\AppData\Local\Temp\ipykernel_27436\2080025358.py:10: LangGraphDeprecatedSinceV11: Accessing GraphOutput via `result[key]` is deprecated. Use `result.value` to access the output value directly, or `result.interrupts` for interrupts. Deprecated in LangGraph V1.1 to be removed in V3.0.
  print(result["messages"][-1].content)



APPROVAL REQUIRED:
{'action_requests': [{'name': 'book_test', 'args': {'slot_id': 17, 'test_id': 6}, 'description': "Tool execution requires approval\n\nTool: book_test\nArgs: {'slot_id': 17, 'test_id': 6}"}], 'review_configs': [{'action_name': 'book_test', 'allowed_decisions': ['approve', 'reject']}]}
Invalid choice. Enter approve or reject.

APPROVAL REQUIRED:
{'action_requests': [{'name': 'book_test', 'args': {'slot_id': 17, 'test_id': 6}, 'description': "Tool execution requires approval\n\nTool: book_test\nArgs: {'slot_id': 17, 'test_id': 6}"}], 'review_configs': [{'action_name': 'book_test', 'allowed_decisions': ['approve', 'reject']}]}
Invalid choice. Enter approve or reject.

APPROVAL REQUIRED:
{'action_requests': [{'name': 'book_test', 'args': {'slot_id': 17, 'test_id': 6}, 'description': "Tool execution requires approval\n\nTool: book_test\nArgs: {'slot_id': 17, 'test_id': 6}"}], 'review_configs': [{'action_name': 'book_test', 'allowed_decisions': ['approve', 'reject']}]}
Inv